# Super Mario Bros PPO Baseline Agent
## Curriculum Learning Research Project

This notebook provides:
- Environment verification and setup
- Baseline PPO agent with full action space
- Configurable hyperparameter framework
- Training monitoring and visualization

**Instructions:**
1. Enable GPU: Runtime → Change runtime type → GPU
2. Run cells sequentially
3. Restart runtime after package installation

## Cell 1: Package Installation
**⚠️ Run this cell, then restart runtime (Runtime → Restart runtime)**

In [2]:
# Install required packages with specific compatible versions
!pip install numpy==1.26.4
!pip install gym-super-mario-bros==7.4.0
!pip install nes-py==8.2.1
!pip install stable-baselines3==2.1.0
!pip install tensorboard
!pip install imageio
!pip install imageio-ffmpeg
!pip install pyglet==1.5.21

print("\n" + "="*60)
print("✓ Installation complete!")
print("⚠️ IMPORTANT: Go to Runtime → Restart runtime")
print("   Then continue with the next cells")
print("="*60)

  Using cached gymnasium-0.29.1-py3-none-any.whl.metadata (10 kB)
Using cached gymnasium-0.29.1-py3-none-any.whl (953 kB)
  Attempting uninstall: gymnasium
    Found existing installation: gymnasium 1.2.2
    Uninstalling gymnasium-1.2.2:
      Successfully uninstalled gymnasium-1.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shimmy 2.0.0 requires gymnasium>=1.0.0a1, but you have gymnasium 0.29.1 which is incompatible.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.1 which is incompatible.



✓ Installation complete!
⚠️ IMPORTANT: Go to Runtime → Restart runtime
   Then continue with the next cells


## Cell 2: Imports and Basic Verification
**Run this after restarting runtime**

In [2]:
import os
import sys
import time
import json
from datetime import datetime
from typing import Dict, List, Optional
import warnings
warnings.filterwarnings('ignore')

# Core libraries
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display, clear_output, Video
from PIL import Image
import imageio

# Use Gymnasium, the successor to Gym, and alias it as gym
import gymnasium as gym
from gymnasium.spaces import Box
from gymnasium.wrappers import GrayScaleObservation, ResizeObservation, FrameStack

# Mario-specific imports
import gym_super_mario_bros
from gym_super_mario_bros.actions import RIGHT_ONLY, SIMPLE_MOVEMENT, COMPLEX_MOVEMENT
from nes_py.wrappers import JoypadSpace

# Stable Baselines 3
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack, VecMonitor, VecTransposeImage
from stable_baselines3.common.callbacks import BaseCallback, EvalCallback, CheckpointCallback
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

print("="*60)
print("✓ All imports successful!")
print("="*60)

# Quick verification
print("\nPackage versions:")
import gymnasium
#print(f"  gymnasium: {gymnasium.__version__}")
#print(f"  gym-super-mario-bros: {gym_super_mario_bros.__version__}")
import stable_baselines3
print(f"  stable-baselines3: {stable_baselines3.__version__}")

✓ All imports successful!

Package versions:
  stable-baselines3: 2.1.0


## Cell 3: Environment Verification
**Verifies that all environment components work correctly**

In [3]:
def verify_environment():
    """Comprehensive environment verification."""
    print("="*60)
    print("ENVIRONMENT VERIFICATION")
    print("="*60)

    # Test 1: Basic environment creation
    print("\n[Test 1] Creating basic environment...")
    try:
        env = gym_super_mario_bros.make('SuperMarioBros-v0')
        print(f"  ✓ Environment created")
        print(f"  • Observation space: {env.observation_space.shape}")
        print(f"  • Action space: {env.action_space}")
        env.close()
    except Exception as e:
        print(f"  ✗ Failed: {e}")
        return False

    # Test 2: Action space wrappers
    print("\n[Test 2] Testing action spaces...")
    action_sets = {
        'RIGHT_ONLY': (RIGHT_ONLY, 5),
        'SIMPLE_MOVEMENT': (SIMPLE_MOVEMENT, 7),
        'COMPLEX_MOVEMENT': (COMPLEX_MOVEMENT, 12)
    }

    for name, (actions, expected_n) in action_sets.items():
        env = gym_super_mario_bros.make('SuperMarioBros-v0')
        env = JoypadSpace(env, actions)
        if env.action_space.n == expected_n:
            print(f"  ✓ {name}: {env.action_space.n} actions")
        else:
            print(f"  ✗ {name}: expected {expected_n}, got {env.action_space.n}")
        env.close()

    # Test 3: Environment stepping
    print("\n[Test 3] Testing environment stepping...")
    try:
        env = gym_super_mario_bros.make('SuperMarioBros-v0')
        env = JoypadSpace(env, SIMPLE_MOVEMENT)
        state = env.reset()

        total_reward = 0
        for step in range(100):
            action = env.action_space.sample()
            state, reward, done, info = env.step(action)
            total_reward += reward
            if done:
                break

        print(f"  ✓ Completed {step+1} steps, total reward: {total_reward:.2f}")
        print(f"  ✓ Info dictionary keys: {list(info.keys())}")
        env.close()
    except Exception as e:
        print(f"  ✗ Failed: {e}")
        return False

    # Test 4: Different levels
    print("\n[Test 4] Testing different levels...")
    test_levels = ['SuperMarioBros-1-1-v0', 'SuperMarioBros-8-1-v0', 'SuperMarioBros-8-4-v0']

    for level in test_levels:
        try:
            env = gym_super_mario_bros.make(level)
            env = JoypadSpace(env, SIMPLE_MOVEMENT)
            state = env.reset()
            print(f"  ✓ {level}")
            env.close()
        except Exception as e:
            print(f"  ✗ {level}: {e}")

    print("\n" + "="*60)
    print("✓ ENVIRONMENT VERIFICATION COMPLETE")
    print("="*60)
    return True

# Run verification
verify_environment()

ENVIRONMENT VERIFICATION

[Test 1] Creating basic environment...
  ✓ Environment created
  • Observation space: (240, 256, 3)
  • Action space: Discrete(256)

[Test 2] Testing action spaces...
  ✓ RIGHT_ONLY: 5 actions
  ✓ SIMPLE_MOVEMENT: 7 actions
  ✓ COMPLEX_MOVEMENT: 12 actions

[Test 3] Testing environment stepping...
  ✓ Completed 100 steps, total reward: 88.00
  ✓ Info dictionary keys: ['coins', 'flag_get', 'life', 'score', 'stage', 'status', 'time', 'world', 'x_pos', 'y_pos']

[Test 4] Testing different levels...
  ✓ SuperMarioBros-1-1-v0
  ✓ SuperMarioBros-8-1-v0
  ✓ SuperMarioBros-8-4-v0

✓ ENVIRONMENT VERIFICATION COMPLETE


True

## Cell 4: Custom Wrappers and Environment Factory

In [10]:
class SkipFrame(gym.Wrapper):
    """Return only every `skip`-th frame, sum rewards."""
    def __init__(self, env, skip=4):
        super().__init__(env)
        self._skip = skip

    def step(self, action):
        total_reward = 0.0
        done = False
        # The step function in gymnasium returns 5 values
        for _ in range(self._skip):
            obs, reward, terminated, truncated, info = self.env.step(action)
            total_reward += reward
            done = terminated or truncated
            if done:
                break
        # Return the 4-tuple that Stable Baselines 3 expects
        return obs, total_reward, done, info


class CustomRewardWrapper(gym.Wrapper):
    """Custom reward wrapper for curriculum learning experiments."""
    def __init__(self, env, x_pos_weight=1.0, time_penalty=0.0,
                 death_penalty=-15.0, coin_reward=0.0, flag_reward=50.0,
                 use_simple_reward=False):
        super().__init__(env)
        self.x_pos_weight = x_pos_weight
        self.time_penalty = time_penalty
        self.death_penalty = death_penalty
        self.coin_reward = coin_reward
        self.flag_reward = flag_reward
        self.use_simple_reward = use_simple_reward
        self._prev_x_pos = 0
        self._prev_coins = 0
        self._prev_time = 400

    def reset(self, **kwargs):
        # Gymnasium's reset returns obs and info
        obs, info = self.env.reset(**kwargs)
        self._prev_x_pos = 0
        self._prev_coins = 0
        self._prev_time = 400
        # Return both obs and info, as expected by DummyVecEnv
        return obs, info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        done = terminated or truncated

        if self.use_simple_reward:
            custom_reward = 0.1 if not done else self.death_penalty
        else:
            custom_reward = 0.0
            x_diff = info['x_pos'] - self._prev_x_pos
            custom_reward += x_diff * self.x_pos_weight
            time_diff = self._prev_time - info['time']
            custom_reward -= time_diff * self.time_penalty
            coin_diff = info['coins'] - self._prev_coins
            custom_reward += coin_diff * self.coin_reward
            if info.get('flag_get', False):
                custom_reward += self.flag_reward
            if done and info.get('life', 3) < 3:
                custom_reward += self.death_penalty
            self._prev_x_pos = info['x_pos']
            self._prev_coins = info['coins']
            self._prev_time = info['time']

        # Return the 4-tuple expected by SB3
        return obs, custom_reward, done, info


def make_mario_env(env_id='SuperMarioBros-v0', action_space='COMPLEX_MOVEMENT',
                   frame_skip=4, grayscale=True, resize_shape=(84, 84),
                   custom_reward=False, reward_kwargs=None, monitor_dir=None):
    """Factory function to create preprocessed Mario environment."""

    action_spaces = {
        'RIGHT_ONLY': RIGHT_ONLY,
        'SIMPLE_MOVEMENT': SIMPLE_MOVEMENT,
        'COMPLEX_MOVEMENT': COMPLEX_MOVEMENT
    }

    # **FIX: Reordered wrappers**
    # 1. Create the core 'gym' environment with its native wrapper first.
    env = gym_super_mario_bros.make(env_id)
    env = JoypadSpace(env, action_spaces[action_space])

    # 2. Apply the compatibility wrapper to make it a Gymnasium env.
    env = gym.wrappers.StepAPICompatibility(env, output_truncation_bool=True)

    # 3. Now, apply all modern 'gymnasium' wrappers.
    if custom_reward:
        reward_kwargs = reward_kwargs or {}
        env = CustomRewardWrapper(env, **reward_kwargs)

    env = SkipFrame(env, skip=frame_skip)

    if grayscale:
        env = GrayScaleObservation(env, keep_dim=True)

    env = ResizeObservation(env, shape=resize_shape)

    # The Monitor wrapper from SB3 is a gymnasium-style wrapper
    if monitor_dir:
        os.makedirs(monitor_dir, exist_ok=True)
        env = Monitor(env, monitor_dir)

    return env


def make_vec_mario_env(env_id='SuperMarioBros-v0', n_envs=1, frame_stack=4, **kwargs):
    """Create vectorized Mario environment with frame stacking."""

    def _make_env():
        return make_mario_env(env_id=env_id, **kwargs)

    env = DummyVecEnv([_make_env for _ in range(n_envs)])
    env = VecMonitor(env)

    # VecFrameStack stacks the frames, resulting in a channels-last format (H, W, C * n_stack).
    env = VecFrameStack(env, n_stack=frame_stack, channels_order='last')

    # VecTransposeImage switches to channels-first (C * n_stack, H, W) for PyTorch.
    env = VecTransposeImage(env)

    return env


# Test the factory
print("Testing environment factory...")
test_env = make_vec_mario_env(
    env_id='SuperMarioBros-1-1-v0',
    action_space='COMPLEX_MOVEMENT',
    frame_stack=4
)
print(f"✓ Created vectorized environment")
print(f"  • Observation shape: {test_env.observation_space.shape}")
print(f"  • Action space: {test_env.action_space}")
test_env.close()

Testing environment factory...


AssertionError: 

## Cell 5: Hyperparameter Configuration
**Modify this cell to change training parameters**

In [5]:
# ============================================================================
# HYPERPARAMETER CONFIGURATION - MODIFY AS NEEDED
# ============================================================================

CONFIG = {
    # ENVIRONMENT SETTINGS
    'env': {
        'env_id': 'SuperMarioBros-v0',  # Full game for baseline
        # For hardest levels: 'SuperMarioBros-8-1-v0', 'SuperMarioBros-8-4-v0'

        'action_space': 'COMPLEX_MOVEMENT',  # 'RIGHT_ONLY', 'SIMPLE_MOVEMENT', 'COMPLEX_MOVEMENT'
        'frame_skip': 4,
        'frame_stack': 4,
        'grayscale': True,
        'resize_shape': (84, 84),
        'n_envs': 8,  # Parallel environments (reduce if OOM)

        # Custom rewards (for curriculum experiments)
        'custom_reward': False,
        'reward_kwargs': {
            'x_pos_weight': 1.0,
            'time_penalty': 0.0,
            'death_penalty': -15.0,
            'coin_reward': 0.0,
            'flag_reward': 50.0,
            'use_simple_reward': False
        }
    },

    # PPO HYPERPARAMETERS
    'ppo': {
        'learning_rate': 2.5e-4,
        'n_steps': 512,          # Steps per env per update
        'batch_size': 256,
        'n_epochs': 4,
        'gamma': 0.99,           # Discount factor
        'gae_lambda': 0.95,
        'clip_range': 0.2,
        'ent_coef': 0.01,        # Entropy coefficient
        'vf_coef': 0.5,          # Value function coefficient
        'max_grad_norm': 0.5
    },

    # TRAINING SETTINGS
    'training': {
        'total_timesteps': 2_000_000,  # Total training steps
        'eval_freq': 50_000,
        'n_eval_episodes': 5,
        'save_freq': 100_000,
        'log_interval': 10,
        'experiment_name': 'baseline_ppo'
    },

    # DIRECTORIES
    'dirs': {
        'model_dir': './models/',
        'log_dir': './logs/',
        'video_dir': './videos/',
        'tensorboard_dir': './tensorboard_logs/'
    }
}

# Print configuration summary
print("="*60)
print("CONFIGURATION SUMMARY")
print("="*60)
print(f"\nEnvironment: {CONFIG['env']['env_id']}")
print(f"Action Space: {CONFIG['env']['action_space']}")
print(f"N Envs: {CONFIG['env']['n_envs']}")
print(f"\nPPO Settings:")
print(f"  Learning Rate: {CONFIG['ppo']['learning_rate']}")
print(f"  N Steps: {CONFIG['ppo']['n_steps']}")
print(f"  Batch Size: {CONFIG['ppo']['batch_size']}")
print(f"\nTraining:")
print(f"  Total Timesteps: {CONFIG['training']['total_timesteps']:,}")
print(f"  Eval Freq: {CONFIG['training']['eval_freq']:,}")
print("="*60)

CONFIGURATION SUMMARY

Environment: SuperMarioBros-v0
Action Space: COMPLEX_MOVEMENT
N Envs: 8

PPO Settings:
  Learning Rate: 0.00025
  N Steps: 512
  Batch Size: 256

Training:
  Total Timesteps: 2,000,000
  Eval Freq: 50,000


## Cell 6: Training Callbacks

In [ ]:
class TrainingMetricsCallback(BaseCallback):
    """Track and plot training metrics."""

    def __init__(self, check_freq=1000, log_dir='./logs/', verbose=1):
        super().__init__(verbose)
        self.check_freq = check_freq
        self.log_dir = log_dir

        self.episode_rewards = []
        self.episode_lengths = []
        self.x_positions = []
        self.completions = []
        self.timestamps = []
        self.avg_rewards = []

        os.makedirs(log_dir, exist_ok=True)

    def _on_step(self) -> bool:
        if len(self.model.ep_info_buffer) > 0:
            for info in self.model.ep_info_buffer:
                if 'r' in info:
                    self.episode_rewards.append(info['r'])
                    self.episode_lengths.append(info['l'])

        infos = self.locals.get('infos', [])
        for info in infos:
            if 'x_pos' in info:
                self.x_positions.append(info['x_pos'])
            if 'flag_get' in info:
                self.completions.append(1 if info['flag_get'] else 0)

        if self.n_calls % self.check_freq == 0:
            self._log_progress()

        return True

    def _log_progress(self):
        if len(self.episode_rewards) == 0:
            return

        recent = self.episode_rewards[-100:]
        avg_reward = np.mean(recent)
        self.avg_rewards.append(avg_reward)
        self.timestamps.append(self.num_timesteps)

        if self.verbose > 0:
            print(f"Step {self.num_timesteps:,} | Avg Reward: {avg_reward:.2f} | "
                  f"Episodes: {len(self.episode_rewards)}")

    def plot_metrics(self, save_path=None):
        """Plot training curves."""
        if len(self.avg_rewards) == 0:
            print("No data to plot yet.")
            return

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        ax1 = axes[0]
        ax1.plot(self.timestamps, self.avg_rewards, 'b-', linewidth=2)
        ax1.set_xlabel('Timesteps')
        ax1.set_ylabel('Average Reward')
        ax1.set_title('Training Reward (100 episode average)')
        ax1.grid(True, alpha=0.3)

        ax2 = axes[1]
        if len(self.episode_rewards) > 0:
            ax2.hist(self.episode_rewards[-500:], bins=50, color='blue', alpha=0.7)
            ax2.set_xlabel('Episode Reward')
            ax2.set_ylabel('Frequency')
            ax2.set_title('Reward Distribution (Last 500 episodes)')

        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"✓ Plot saved to {save_path}")

        plt.show()


class SaveBestCallback(BaseCallback):
    """Save model when best reward is achieved."""

    def __init__(self, check_freq=1000, save_path='./models/best_model', verbose=1):
        super().__init__(verbose)
        self.check_freq = check_freq
        self.save_path = save_path
        self.best_mean_reward = -np.inf

    def _on_step(self) -> bool:
        if self.n_calls % self.check_freq == 0:
            if len(self.model.ep_info_buffer) > 0:
                rewards = [ep['r'] for ep in self.model.ep_info_buffer]
                mean_reward = np.mean(rewards)

                if mean_reward > self.best_mean_reward:
                    self.best_mean_reward = mean_reward
                    self.model.save(self.save_path)
                    if self.verbose > 0:
                        print(f"  ✓ New best! Mean reward: {mean_reward:.2f}")

        return True

print("✓ Callbacks defined!")

## Cell 7: Training Function

In [ ]:
def train_baseline_agent(config=CONFIG, resume_from=None):
    """
    Train the baseline PPO agent.

    Parameters:
    -----------
    config : dict
        Configuration dictionary
    resume_from : str, optional
        Path to model to resume from

    Returns:
    --------
    tuple : (model, metrics_callback)
    """

    print("="*60)
    print("TRAINING BASELINE PPO AGENT")
    print("="*60)

    env_cfg = config['env']
    ppo_cfg = config['ppo']
    train_cfg = config['training']
    dirs_cfg = config['dirs']

    # Create directories
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    exp_name = f"{train_cfg['experiment_name']}_{timestamp}"

    model_dir = os.path.join(dirs_cfg['model_dir'], exp_name)
    log_dir = os.path.join(dirs_cfg['log_dir'], exp_name)

    for d in [model_dir, log_dir]:
        os.makedirs(d, exist_ok=True)

    # Save config
    with open(os.path.join(model_dir, 'config.json'), 'w') as f:
        json.dump(config, f, indent=2)

    print(f"\n[1] Creating training environment...")
    print(f"  • Environment: {env_cfg['env_id']}")
    print(f"  • Action Space: {env_cfg['action_space']}")
    print(f"  • N Environments: {env_cfg['n_envs']}")

    # Create environments
    train_env = make_vec_mario_env(
        env_id=env_cfg['env_id'],
        n_envs=env_cfg['n_envs'],
        action_space=env_cfg['action_space'],
        frame_skip=env_cfg['frame_skip'],
        frame_stack=env_cfg['frame_stack'],
        grayscale=env_cfg['grayscale'],
        resize_shape=env_cfg['resize_shape'],
        custom_reward=env_cfg['custom_reward'],
        reward_kwargs=env_cfg['reward_kwargs'],
        monitor_dir=os.path.join(log_dir, 'train_monitor')
    )

    print(f"  ✓ Training environment: {train_env.observation_space.shape}")

    eval_env = make_vec_mario_env(
        env_id=env_cfg['env_id'],
        n_envs=1,
        action_space=env_cfg['action_space'],
        frame_skip=env_cfg['frame_skip'],
        frame_stack=env_cfg['frame_stack'],
        grayscale=env_cfg['grayscale'],
        resize_shape=env_cfg['resize_shape']
    )

    # Callbacks
    print(f"\n[2] Setting up callbacks...")

    metrics_callback = TrainingMetricsCallback(
        check_freq=5000,
        log_dir=log_dir,
        verbose=1
    )

    checkpoint_callback = CheckpointCallback(
        save_freq=train_cfg['save_freq'] // env_cfg['n_envs'],
        save_path=model_dir,
        name_prefix='mario_ppo',
        verbose=1
    )

    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path=os.path.join(model_dir, 'best'),
        log_path=log_dir,
        eval_freq=train_cfg['eval_freq'] // env_cfg['n_envs'],
        n_eval_episodes=train_cfg['n_eval_episodes'],
        deterministic=True,
        verbose=1
    )

    best_callback = SaveBestCallback(
        check_freq=10000,
        save_path=os.path.join(model_dir, 'best_training'),
        verbose=1
    )

    callbacks = [metrics_callback, checkpoint_callback, eval_callback, best_callback]
    print(f"  ✓ Callbacks configured")

    # Create model
    print(f"\n[3] Creating PPO model...")

    if resume_from and os.path.exists(resume_from):
        print(f"  • Resuming from: {resume_from}")
        model = PPO.load(resume_from, env=train_env,
                        tensorboard_log=dirs_cfg['tensorboard_dir'])
    else:
        model = PPO(
            policy='CnnPolicy',
            env=train_env,
            learning_rate=ppo_cfg['learning_rate'],
            n_steps=ppo_cfg['n_steps'],
            batch_size=ppo_cfg['batch_size'],
            n_epochs=ppo_cfg['n_epochs'],
            gamma=ppo_cfg['gamma'],
            gae_lambda=ppo_cfg['gae_lambda'],
            clip_range=ppo_cfg['clip_range'],
            ent_coef=ppo_cfg['ent_coef'],
            vf_coef=ppo_cfg['vf_coef'],
            max_grad_norm=ppo_cfg['max_grad_norm'],
            verbose=1,
            tensorboard_log=dirs_cfg['tensorboard_dir'],
            device='auto'
        )

    print(f"  ✓ PPO model on device: {model.device}")

    # Train
    print(f"\n[4] Starting training...")
    print(f"  • Total timesteps: {train_cfg['total_timesteps']:,}")
    print("-"*60)

    try:
        model.learn(
            total_timesteps=train_cfg['total_timesteps'],
            callback=callbacks,
            log_interval=train_cfg['log_interval'],
            tb_log_name=exp_name,
            reset_num_timesteps=resume_from is None,
            progress_bar=True
        )

        final_path = os.path.join(model_dir, 'final_model')
        model.save(final_path)
        print(f"\n✓ Training complete! Model saved to: {final_path}")

    except KeyboardInterrupt:
        print("\n⚠ Training interrupted")
        interrupt_path = os.path.join(model_dir, 'interrupted_model')
        model.save(interrupt_path)
        print(f"  Model saved to: {interrupt_path}")

    finally:
        train_env.close()
        eval_env.close()

    return model, metrics_callback, model_dir

print("✓ Training function defined!")

## Cell 8: Evaluation and Video Recording

In [ ]:
def evaluate_and_record(model_path, env_id='SuperMarioBros-v0',
                        action_space='COMPLEX_MOVEMENT', n_episodes=5,
                        video_path='./videos/evaluation.mp4'):
    """
    Evaluate agent and record video.
    """

    print("="*60)
    print("EVALUATING AGENT")
    print("="*60)

    # Load model
    model = PPO.load(model_path)
    print(f"✓ Model loaded from: {model_path}")

    # Create eval environment
    eval_env = make_vec_mario_env(
        env_id=env_id,
        n_envs=1,
        action_space=action_space,
        frame_skip=4,
        frame_stack=4,
        grayscale=True,
        resize_shape=(84, 84)
    )

    # Video env (for recording)
    action_spaces = {
        'RIGHT_ONLY': RIGHT_ONLY,
        'SIMPLE_MOVEMENT': SIMPLE_MOVEMENT,
        'COMPLEX_MOVEMENT': COMPLEX_MOVEMENT
    }
    video_env = gym_super_mario_bros.make(env_id)
    video_env = JoypadSpace(video_env, action_spaces[action_space])
    video_env = SkipFrame(video_env, skip=4)

    # Evaluation metrics
    results = {'rewards': [], 'lengths': [], 'x_pos': [], 'completed': []}
    frames = []

    print(f"\nRunning {n_episodes} episodes...")

    for ep in range(n_episodes):
        obs = eval_env.reset()
        video_obs = video_env.reset()

        ep_reward = 0
        ep_length = 0
        done = False

        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, info = eval_env.step(action)
            video_obs, _, _, video_info = video_env.step(action[0])

            ep_reward += reward[0]
            ep_length += 1

            if ep == 0:  # Record first episode
                frames.append(video_obs)

            if done[0]:
                break

        results['rewards'].append(ep_reward)
        results['lengths'].append(ep_length)
        results['x_pos'].append(info[0].get('x_pos', 0))
        results['completed'].append(info[0].get('flag_get', False))

        print(f"  Ep {ep+1}: Reward={ep_reward:.1f}, X_pos={results['x_pos'][-1]}")

    # Statistics
    print(f"\n" + "="*60)
    print("RESULTS")
    print("="*60)
    print(f"Mean Reward: {np.mean(results['rewards']):.2f} ± {np.std(results['rewards']):.2f}")
    print(f"Mean X Position: {np.mean(results['x_pos']):.0f}")
    print(f"Completion Rate: {np.mean(results['completed'])*100:.1f}%")

    # Save video
    if len(frames) > 0:
        os.makedirs(os.path.dirname(video_path) if os.path.dirname(video_path) else '.', exist_ok=True)
        imageio.mimsave(video_path, frames, fps=30)
        print(f"\n✓ Video saved to: {video_path}")

    eval_env.close()
    video_env.close()

    return results


def display_video(video_path):
    """Display video in Colab."""
    from IPython.display import Video
    return Video(video_path, embed=True, width=600)

print("✓ Evaluation functions defined!")

## Cell 9: Quick Test (Short Training Run)
**Run this to verify everything works before full training**

In [ ]:
# Quick test with reduced timesteps
# This should take ~5-10 minutes

QUICK_CONFIG = CONFIG.copy()
QUICK_CONFIG['training'] = {
    'total_timesteps': 100_000,  # Reduced for quick test
    'eval_freq': 25_000,
    'n_eval_episodes': 3,
    'save_freq': 50_000,
    'log_interval': 10,
    'experiment_name': 'quick_test'
}
QUICK_CONFIG['env']['n_envs'] = 4  # Reduced

print("Starting quick test training (100k steps)...")
print("This should take approximately 5-10 minutes.\n")

model, metrics, model_dir = train_baseline_agent(QUICK_CONFIG)

In [ ]:
# Plot training metrics
metrics.plot_metrics()

## Cell 10: Full Baseline Training
**Run this for full training (2M steps, ~2-4 hours with GPU)**

In [ ]:
# Full baseline training
# WARNING: This takes 2-4 hours with GPU!

# Uncomment to run:
# model, metrics, model_dir = train_baseline_agent(CONFIG)

## Cell 11: Evaluate Trained Agent

In [ ]:
# Evaluate the trained model
# Replace with your model path

model_path = f"{model_dir}/final_model.zip"  # or best/best_model.zip

results = evaluate_and_record(
    model_path=model_path,
    env_id='SuperMarioBros-1-1-v0',  # Test on 1-1
    n_episodes=5,
    video_path='./videos/eval_1-1.mp4'
)

In [ ]:
# Display the recorded video
display_video('./videos/eval_1-1.mp4')

## Cell 12: TensorBoard (Training Visualization)

In [ ]:
# Load TensorBoard extension
%load_ext tensorboard

# View training logs
%tensorboard --logdir ./tensorboard_logs/

## Cell 13: Mount Google Drive (Optional - Save Models Permanently)

In [ ]:
# Mount Google Drive to save models permanently
from google.colab import drive
drive.mount('/content/drive')

# Update paths to save to Google Drive
CONFIG['dirs']['model_dir'] = '/content/drive/MyDrive/mario_rl/models/'
CONFIG['dirs']['log_dir'] = '/content/drive/MyDrive/mario_rl/logs/'
CONFIG['dirs']['video_dir'] = '/content/drive/MyDrive/mario_rl/videos/'

print("✓ Google Drive mounted!")
print(f"Models will be saved to: {CONFIG['dirs']['model_dir']}")

---

## Curriculum Learning Extensions

For your curriculum learning experiments, modify the CONFIG:

### Action Space Curriculum
```python
# Stage 1: RIGHT_ONLY (5 actions)
CONFIG['env']['action_space'] = 'RIGHT_ONLY'

# Stage 2: SIMPLE_MOVEMENT (7 actions)  
CONFIG['env']['action_space'] = 'SIMPLE_MOVEMENT'

# Stage 3: COMPLEX_MOVEMENT (12 actions)
CONFIG['env']['action_space'] = 'COMPLEX_MOVEMENT'
```

### Reward Curriculum
```python
# Stage 1: Survival only
CONFIG['env']['custom_reward'] = True
CONFIG['env']['reward_kwargs']['use_simple_reward'] = True

# Stage 2: Add movement reward
CONFIG['env']['reward_kwargs']['use_simple_reward'] = False
CONFIG['env']['reward_kwargs']['x_pos_weight'] = 1.0

# Stage 3: Full reward
CONFIG['env']['reward_kwargs']['coin_reward'] = 1.0
CONFIG['env']['reward_kwargs']['flag_reward'] = 100.0
```

### Level Curriculum
```python
# Easy → Hard levels
levels = [
    'SuperMarioBros-1-1-v0',  # Easiest
    'SuperMarioBros-2-1-v0',
    'SuperMarioBros-4-1-v0',
    'SuperMarioBros-6-1-v0',
    'SuperMarioBros-8-1-v0',  # Hardest
]
```